In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName('unit_test') \
    .config("spark.jars", "/opt/spark/jars/iceberg-spark-runtime-3.5_2.12-1.6.0.jar") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.spark_catalog.type", "hive") \
    .config("spark.sql.catalog.local.warehouse", "s3a://datalake/iceberg") \
    .getOrCreate()

#Ajuste de log WARN log para ERROR
spark.sparkContext.setLogLevel("ERROR")

25/11/16 15:49:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/16 15:49:31 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/11/16 15:49:31 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
25/11/16 15:49:31 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
25/11/16 15:49:31 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.
25/11/16 15:49:31 WARN Utils: Service 'SparkUI' could not bind on port 4044. Attempting port 4045.


In [2]:
import unittest

class DataQualityTestSuite(unittest.TestCase):
    def __init__(self, methodName='runTest', target_table=None):
        super().__init__(methodName)
        self.target_table = target_table

    def test_returned_orders(self):
        """Test the percentage of returned orders"""
        src = spark.sql(f"""
            SELECT 
                ROUND(
                    COUNT(DISTINCT CASE WHEN is_returned = 'true' THEN order_id END) / 
                    COUNT(DISTINCT order_id),
                2) AS returned_percent
            FROM 
                {self.target_table}
        """).take(1)[0][0]

        expected = 0.02
        self.assertEqual(float(src), expected, f" Percentual devolução é diferente do esperado")


In [3]:
# Fábrica para criar uma nova classe de teste com o target_table embutido (injeção de dependência com class factory)

def create_test_suite_for_table(target_table):
    
    class PatchedTestSuite(DataQualityTestSuite):
        def __init__(self, methodName='runTest'):
            super().__init__(methodName=methodName, target_table=target_table)
            
    PatchedTestSuite.__name__ = f"DataQualityTestSuite_{target_table.replace('.', '_')}"
    
    return unittest.TestLoader().loadTestsFromTestCase(PatchedTestSuite)


In [4]:
target_table='iceberg.bronze.tbl_bronze_order_events'

metrics_quality_suite = create_test_suite_for_table(target_table)
tests = unittest.TestSuite([metrics_quality_suite])
runner = unittest.TextTestRunner(verbosity=2)
runner.run(tests)

test_returned_orders (__main__.create_test_suite_for_table.<locals>.PatchedTestSuite)
Test the percentage of returned orders ... SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
ok                                                                              

----------------------------------------------------------------------
Ran 1 test in 17.149s

OK


<unittest.runner.TextTestResult run=1 errors=0 failures=0>

In [ ]:
spark.stop()